# Trabalho 1 — Aquisição de Dados
## Enchentes de 2024 no Rio Grande do Sul × Indicadores Econômicos (IBGE)

**Disciplina:** Ciência de Dados — Instituto de Computação (UFAM)

**Data da coleta:** gerada automaticamente — ver coluna `data_hora_coleta_utc` em `registro_proveniencia.csv` (Seção 8)

---

### Fontes utilizadas

| # | Fonte | Método | O que fornece |
|---|-------|--------|----------------|
| 1 | **Boletins/notícias sobre as enchentes de 2024** (Defesa Civil RS via imprensa) | **Web scraping** (`requests` + `BeautifulSoup`) | Nº de municípios afetados, óbitos, desaparecidos, desalojados, pessoas em abrigos e nível do Guaíba, por data de boletim |
| 2 | **API do IBGE** (Localidades + SIDRA/Agregados) | **API REST** | Lista dos 497 municípios do RS (código e nome); **IPCA mensal** e **taxa de desocupação** (PNAD Contínua) |

**Chave de integração:** `ano_mes` (`AAAA-MM`). A granularidade diária dos boletins é convertida
para mensal antes do merge (detalhes na Seção 6).

### Decisões metodológicas importantes (leia antes de rodar)

- **IPCA e Porto Alegre.** Diferente de outras capitais do Norte, **Porto Alegre é uma das áreas de
  coleta históricas do IPCA** (uma das 16 regiões metropolitanas cobertas desde antes de 2020), então
  aqui **não é necessário usar um proxy regional** — o notebook procura Porto Alegre diretamente na
  tabela e cai para um proxy (Região Sul, depois Brasil) só se, por algum motivo, ela não aparecer.
  A localidade realmente usada fica registrada na coluna `ipca_localidade` da base, para o grupo
  conferir.
- **Períodos explícitos.** Os períodos são calculados a partir de `JANELA_INICIO`/`JANELA_FIM`
  (Seção 1). **Não** usamos o atalho relativo `-N` da API, que devolve "os últimos N períodos" na
  data de execução e deixaria a coleta fora da janela da enchente.
- **Desocupação trimestral.** A PNAD Contínua trimestral tem códigos `AAAA01`–`AAAA04`
  (trimestres), que **não** são meses. O código converte cada trimestre nos seus 3 meses
  (o valor do trimestre é repetido nos meses) e guarda o período original em
  `desocupacao_periodo_original`.
- **Valores manuais têm prioridade sobre os automáticos** (o regex sobre texto jornalístico pode
  errar), e cada valor tem uma coluna `origem_*` indicando de onde veio. Divergências entre
  automático e manual são listadas na Seção 6.1.

### Observação sobre robustez

Páginas de notícias mudam de estrutura com o tempo. As células de scraping são defensivas
(try/except, regex tolerante, validação de faixa de valores), **gravam o HTML bruto (bytes originais)
antes de qualquer extração** e registram o hash SHA-256 no log de proveniência. Se algum site
retornar erro (403, timeout), ajuste a URL ou use a conferência manual (Seção 3).

> **Nota:** os boletins oficiais da Defesa Civil do RS (`estado.rs.gov.br`) publicam parte dos
> números como **cartão/infográfico** (imagem), não como texto puro em todas as páginas — por isso
> priorizamos aqui **matérias de portais de notícia** (Sul21, Correio Braziliense, IHU/Unisinos,
> Diário do Nordeste, Revista Oeste) que republicam os mesmos números em texto corrido, mais
> confiável para `BeautifulSoup` + regex. Se quiser raspar o portal oficial diretamente, será
> necessário OCR ou extração de PDF (fora do escopo de scraping de HTML deste trabalho).


## 1. Setup do ambiente

In [ ]:
# Bibliotecas — no Google Colab, requests/bs4/pandas/lxml já vêm instaladas.
import os
import re
import json
import time
import hashlib
import zipfile
import datetime as dt
import urllib.robotparser as robotparser
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

try:
    from unidecode import unidecode
except ImportError:
    %pip install -q unidecode
    from unidecode import unidecode

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Ambiente pronto.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.4 MB/s eta 0:00:00
Ambiente pronto.


In [ ]:
# ======================= CONFIGURAÇÃO (edite aqui) =======================
CONTATO_EMAIL = " "
USER_AGENT_NOME = "TesteAcademico"

# Janela temporal da análise (inclusive). Cobre antes/durante/depois da enchente de 2024,
# incluindo um ponto de checagem ~1 ano depois (balanço de abril/2025).
JANELA_INICIO = "2023-06"
JANELA_FIM = "2025-06"

# Regra (ARBITRÁRIA, documentada) para o rótulo "em_crise": >= N municípios afetados (de 497 no RS).
LIMIAR_CRISE_MUNICIPIOS = 300

DELAY_ENTRE_REQUISICOES = 2            # segundos (item 3.5 do enunciado)
ROBOTS_INDETERMINADO_PROSSEGUIR = False  # se robots.txt não puder ser lido: True = raspa mesmo assim

# IDs de tabelas do SIDRA (conferidos por nome/metadados na Seção 4; troque aqui se necessário)
ID_TABELA_IPCA = 7060           # IPCA — variação mensal etc. (a partir de jan/2020)
ID_TABELA_DESOCUPACAO = 4099    # PNAD Contínua trimestral — taxa de desocupação (14 anos ou mais)

# Colab tem disco efêmero: para não perder tudo ao desconectar, monte o Drive.
MONTAR_DRIVE = False
INCLUIR_HTML_BRUTO_NO_ZIP = False   # HTML de matérias é conteúdo protegido: por padrão NÃO vai no zip

# ==========================================================================
BASE_DIR = "."
if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/trabalho1_enchentes"

DIR_BRUTOS = os.path.join(BASE_DIR, "dados_brutos")
DIR_BRUTOS_SCRAPING = os.path.join(DIR_BRUTOS, "scraping_enchentes")
DIR_BRUTOS_IBGE = os.path.join(DIR_BRUTOS, "api_ibge")
DIR_TRATADOS = os.path.join(BASE_DIR, "dados_tratados")

for d in [DIR_BRUTOS, DIR_BRUTOS_SCRAPING, DIR_BRUTOS_IBGE, DIR_TRATADOS]:
    os.makedirs(d, exist_ok=True)

HEADERS = {
    "User-Agent": (
        f"Mozilla/5.0 (compatible; {USER_AGENT_NOME}/1.1; "
        f"uso academico, sem fins comerciais; contato: {CONTATO_EMAIL})"
    )
}
if "SEU_EMAIL_AQUI" in CONTATO_EMAIL:
    print("[aviso] Preencha CONTATO_EMAIL: o User-Agent deve permitir que os sites contatem os autores.")

MESES_JANELA = list(pd.period_range(JANELA_INICIO, JANELA_FIM, freq="M").strftime("%Y-%m"))
print("Pastas:", DIR_BRUTOS, "|", DIR_TRATADOS)
print(f"Janela de análise: {MESES_JANELA[0]} a {MESES_JANELA[-1]} ({len(MESES_JANELA)} meses)")


Pastas: ./dados_brutos | ./dados_tratados
Janela de análise: 2023-06 a 2025-06 (25 meses)


In [ ]:
# --- Utilitários gerais: proveniência (item 3.4), hash, HTTP com retry, números pt-BR ----------
provenance_log = []


def agora_utc():
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")


def sha256_bytes(conteudo: bytes) -> str:
    return hashlib.sha256(conteudo).hexdigest()


def registrar_provenancia(fonte, url, metodo, parametros=None, observacao="", arquivo=None, sha256=None):
    '''Anota fonte, URL exata, data/hora, método, parâmetros e (se houver) arquivo bruto + hash.'''
    registro = {
        "fonte": fonte,
        "url": url,
        "data_hora_coleta_utc": agora_utc(),
        "metodo": metodo,
        "parametros": json.dumps(parametros or {}, ensure_ascii=False),
        "observacao": observacao,
        "arquivo_bruto": arquivo,
        "sha256": sha256,
    }
    provenance_log.append(registro)
    return registro


def salvar_bruto(caminho, conteudo: bytes) -> str:
    '''Grava os bytes EXATOS recebidos e devolve o SHA-256.'''
    with open(caminho, "wb") as f:
        f.write(conteudo)
    return sha256_bytes(conteudo)


def salvar_json_bruto(caminho, obj) -> str:
    return salvar_bruto(caminho, json.dumps(obj, ensure_ascii=False, indent=2).encode("utf-8"))


def http_get(url, tentativas=3, timeout=30, espera=2, **kwargs):
    '''GET com retry/backoff para erros de rede e HTTP 429/5xx. Pode devolver resposta 4xx.'''
    ultimo_erro = None
    for i in range(1, tentativas + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=timeout, **kwargs)
            if resp.status_code in (429, 500, 502, 503, 504) and i < tentativas:
                time.sleep(espera * i)
                continue
            return resp
        except requests.exceptions.RequestException as e:
            ultimo_erro = e
            if i < tentativas:
                time.sleep(espera * i)
    raise ultimo_erro


def parse_num_ptbr(txt):
    '''Converte número em texto para float, tratando pt-BR ('1.234,5') e ponto decimal ('12.89').
    - '12,89' -> 12.89 | '12.89' -> 12.89 | '1.289' -> 1289 (milhar) | '1.234,5' -> 1234.5'''
    s = str(txt).strip()
    if not s:
        return None
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    elif "." in s and re.fullmatch(r"\d{1,3}(\.\d{3})+", s):
        s = s.replace(".", "")
    try:
        return float(s)
    except ValueError:
        return None


print("Utilitários prontos.")


Utilitários prontos.


## 2. Verificação de `robots.txt` (postura ética — item 3.5)

Antes de raspar, checamos o `robots.txt` **de cada URL que será acessada** (não só da raiz do
domínio), usando nosso próprio User-Agent. Resultado possível:

- `permitido` — pode raspar;
- `bloqueado` — a URL é **pulada** pelo laço de scraping;
- `indeterminado` — não foi possível ler o `robots.txt` (erro de rede, 401/403, 5xx). Por
  padrão (postura conservadora) a URL também é **pulada**; para prosseguir por conta e risco do
  grupo, mude `ROBOTS_INDETERMINADO_PROSSEGUIR` na Seção 1 e registre a decisão no dataset card.

Se o `robots.txt` declarar `Crawl-delay`, o maior entre ele e `DELAY_ENTRE_REQUISICOES` é usado.
Cada leitura de `robots.txt` entra no log de proveniência.


In [ ]:
_robots_cache = {}


def _carregar_robots(url_alvo):
    p = urlparse(url_alvo)
    base = f"{p.scheme}://{p.netloc}"
    if base in _robots_cache:
        return _robots_cache[base]

    robots_url = f"{base}/robots.txt"
    info = {"robots_url": robots_url, "parser": None, "status": "indeterminado", "detalhe": ""}
    try:
        resp = http_get(robots_url, tentativas=2, timeout=15)
        if resp.status_code == 200:
            rp = robotparser.RobotFileParser()
            rp.parse(resp.text.splitlines())
            info.update(parser=rp, status="ok", detalhe="HTTP 200")
        elif resp.status_code in (404, 410):
            info.update(status="sem_robots", detalhe=f"HTTP {resp.status_code}: sem robots.txt")
        else:  # 401/403/429/5xx...: não dá para saber a política do site
            info.update(status="indeterminado", detalhe=f"HTTP {resp.status_code}")
    except Exception as e:
        info["detalhe"] = f"erro de rede: {e}"

    registrar_provenancia(
        fonte=f"robots.txt de {p.netloc}", url=robots_url,
        metodo="GET (requests + urllib.robotparser)", parametros={"user_agent": USER_AGENT_NOME},
        observacao=f"status={info['status']}; {info['detalhe']}",
    )
    _robots_cache[base] = info
    return info


def checar_robots(url_alvo):
    '''Retorna dict com 'pode' ('permitido'|'bloqueado'|'indeterminado'), 'crawl_delay' e detalhes.'''
    info = _carregar_robots(url_alvo)
    delay = None
    if info["status"] == "ok":
        rp = info["parser"]
        pode = "permitido" if rp.can_fetch(USER_AGENT_NOME, url_alvo) else "bloqueado"
        delay = rp.crawl_delay(USER_AGENT_NOME)
    elif info["status"] == "sem_robots":
        pode = "permitido"
    else:
        pode = "indeterminado"
    return {"pode": pode, "crawl_delay": delay, "robots_url": info["robots_url"], "detalhe": info["detalhe"]}


print("Checagem de robots.txt pronta.")


Checagem de robots.txt pronta.


## 3. Fonte 1 — Web Scraping: boletins das enchentes de 2024 no Rio Grande do Sul

A Defesa Civil do RS divulgou boletins diários (às vezes 2 a 3 por dia) durante as enchentes de
abril/maio de 2024, informando: número de municípios afetados, óbitos confirmados, desaparecidos,
pessoas desalojadas, pessoas em abrigos e, em algumas coberturas, o nível do Guaíba em Porto Alegre —
o principal indicador visual da enchente na capital. Esses boletins não têm API pública, mas foram
republicados em páginas HTML de portais de notícia (com destaque para o `sul21.com.br`, que cobriu
quase diariamente a evolução do desastre), que é o que raspamos aqui.

> **Limitação conhecida:** cada matéria cobre uma data específica; a série resultante é **concentrada
> em maio de 2024**, com um único ponto de checagem um ano depois (abril/2025). Também há
> republicação de números oficiais por veículos diferentes no mesmo dia, então algumas linhas
> **não são observações independentes**. Ver Seção 11 (próximos passos).

**Confira cada URL manualmente** (título, data, teor) antes de usar; ajuste/substitua livremente por
fontes equivalentes, de preferência **primárias** (Defesa Civil RS, Governo do RS, ANA/CPRM).


In [ ]:
# Lista curada de páginas com boletins/números das enchentes de 2024 no RS.
# data_ref = data de referência atribuída pelo grupo (será comparada com a data publicada na página).
URLS_ENCHENTES = [
    {"data_ref": "2024-05-01", "fonte": "Sul21",
     "url": "https://sul21.com.br/?p=235018"},
    {"data_ref": "2024-05-02", "fonte": "Sul21",
     "url": "https://sul21.com.br/?p=235344"},
    {"data_ref": "2024-05-02", "fonte": "Diário do Nordeste",
     "url": "https://diariodonordeste.verdesmares.com.br/ultima-hora/pais/rio-grande-do-sul-tem-29-mortos-em-chuvas-atualiza-defesa-civil-1.3507808"},
    {"data_ref": "2024-05-03", "fonte": "Sul21",
     "url": "https://sul21.com.br/?p=235669"},
    {"data_ref": "2024-05-08", "fonte": "Sul21",
     "url": "https://sul21.com.br/?p=237057"},
    {"data_ref": "2024-05-13", "fonte": "IHU/Unisinos",
     "url": "https://ihu.unisinos.br/639521-extensao-territorial-e-numero-de-afetados-tornam-tragedia-no-rs-inedita-no-brasil"},
    {"data_ref": "2024-05-16", "fonte": "Sul21",
     "url": "https://sul21.com.br/?p=238281"},
    {"data_ref": "2024-05-17", "fonte": "Sul21",
     "url": "https://sul21.com.br/?p=238419"},
    {"data_ref": "2024-05-17", "fonte": "Correio Braziliense",
     "url": "https://www.correiobraziliense.com.br/brasil/2024/05/6859198-mais-de-90-dos-municipios-gauchos-sao-atingidos-pela-tragedia-climatica-no-sul.html"},
    {"data_ref": "2025-04-24", "fonte": "Revista Oeste",
     "url": "https://www.revistaoeste.com/brasil/rio-grande-do-sul-confirma-184-vitima-das-enchentes-de-2024/"},
]

df_urls = pd.DataFrame(URLS_ENCHENTES)
df_urls


,data_ref,fonte,url
0,2024-05-01,Sul21,https://sul21.com.br/?p=235018
1,2024-05-02,Sul21,https://sul21.com.br/?p=235344
2,2024-05-02,Diário do Nordeste,https://diariodonordeste.verdesmares.com.br/ul...
3,2024-05-03,Sul21,https://sul21.com.br/?p=235669
4,2024-05-08,Sul21,https://sul21.com.br/?p=237057
5,2024-05-13,IHU/Unisinos,https://ihu.unisinos.br/639521-extensao-territ...
6,2024-05-16,Sul21,https://sul21.com.br/?p=238281
7,2024-05-17,Sul21,https://sul21.com.br/?p=238419
8,2024-05-17,Correio Braziliense,https://www.correiobraziliense.com.br/brasil/2...
9,2025-04-24,Revista Oeste,https://www.revistaoeste.com/brasil/rio-grande...


In [ ]:
# Checagem de robots.txt de CADA URL (com o nosso User-Agent). O resultado é usado pelo laço de scraping.
robots_resultado = []
for item in URLS_ENCHENTES:
    r = checar_robots(item["url"])
    item["robots"] = r["pode"]
    item["crawl_delay"] = r["crawl_delay"]
    robots_resultado.append({
        "dominio": urlparse(item["url"]).netloc, "url": item["url"], "robots_url": r["robots_url"],
        "resultado": r["pode"], "crawl_delay": r["crawl_delay"], "detalhe": r["detalhe"],
    })

df_robots = pd.DataFrame(robots_resultado)
print(df_robots["resultado"].value_counts().to_string())
df_robots


resultado
permitido    10


,dominio,url,robots_url,resultado,crawl_delay,detalhe
0,sul21.com.br,https://sul21.com.br/?p=235018,https://sul21.com.br/robots.txt,permitido,NaN,HTTP 200
1,sul21.com.br,https://sul21.com.br/?p=235344,https://sul21.com.br/robots.txt,permitido,NaN,HTTP 200
2,diariodonordeste.verdesmares.com.br,https://diariodonordeste.verdesmares.com.br/ul...,https://diariodonordeste.verdesmares.com.br/ro...,permitido,NaN,HTTP 200
3,sul21.com.br,https://sul21.com.br/?p=235669,https://sul21.com.br/robots.txt,permitido,NaN,HTTP 200
4,sul21.com.br,https://sul21.com.br/?p=237057,https://sul21.com.br/robots.txt,permitido,NaN,HTTP 200
5,ihu.unisinos.br,https://ihu.unisinos.br/639521-extensao-territ...,https://ihu.unisinos.br/robots.txt,permitido,NaN,HTTP 200
6,sul21.com.br,https://sul21.com.br/?p=238281,https://sul21.com.br/robots.txt,permitido,NaN,HTTP 200
7,sul21.com.br,https://sul21.com.br/?p=238419,https://sul21.com.br/robots.txt,permitido,NaN,HTTP 200
8,www.correiobraziliense.com.br,https://www.correiobraziliense.com.br/brasil/2...,https://www.correiobraziliense.com.br/robots.txt,permitido,NaN,HTTP 200
9,www.revistaoeste.com,https://www.revistaoeste.com/brasil/rio-grande...,https://www.revistaoeste.com/robots.txt,permitido,10.0,HTTP 200


**Leitura do resultado acima:** URLs `bloqueado` são puladas. URLs `indeterminado` também são puladas
(a menos que `ROBOTS_INDETERMINADO_PROSSEGUIR = True`). Erros 401/403 na leitura do `robots.txt`
costumam vir de proteção anti-bot (ex.: Cloudflare), não de uma política do site — nesses casos,
use a conferência manual (abaixo) ou decida conscientemente prosseguir e **registre a decisão**.


In [ ]:
def extrair_texto_pagina(html_bytes):
    '''Devolve (titulo, data_publicacao_meta, texto_do_conteudo_principal).
    Foca no <article>/<main> e remove blocos de ruído (relacionadas, comentários, compartilhar...)
    para não capturar números de OUTRAS matérias listadas na página.'''
    def _sopa():
        return BeautifulSoup(html_bytes, "lxml")

    soup = _sopa()
    titulo = soup.title.get_text(strip=True) if soup.title else None

    data_meta = None
    for attrs in ({"property": "article:published_time"}, {"name": "article:published_time"},
                  {"itemprop": "datePublished"}, {"name": "date"}, {"property": "og:updated_time"}):
        tag = soup.find("meta", attrs=attrs)
        if tag is not None and tag.get("content"):
            data_meta = tag["content"][:10]
            break
    if data_meta is None:
        t = soup.find("time", attrs={"datetime": True})
        if t is not None:
            data_meta = t["datetime"][:10]

    # Limpeza principal (NÃO remove <header>: o lead/título da matéria costuma estar nele)
    for tag in soup(["script", "style", "noscript", "nav", "footer", "aside", "form", "iframe"]):
        tag.decompose()
    ruido = re.compile(
        r"relacionad|leia-?mais|leia-?tamb|related|comment|coment|share|compartilh|"
        r"newsletter|sidebar|widget|publicidade|advert", re.I)
    for tag in soup.find_all(attrs={"class": ruido}) + soup.find_all(attrs={"id": ruido}):
        if not getattr(tag, "decomposed", False):
            tag.decompose()

    area = soup.find("article") or soup.find("main") or soup.body or soup
    texto = re.sub(r"\s+", " ", area.get_text(separator=" ")).strip()

    # Se a limpeza por classe apagou quase tudo (classe de "ruído" em um contêiner grande), refaz
    # SEM o filtro por classe/id, mas ainda sem menu/rodapé/lateral, e avisa no texto de retorno.
    if len(texto) < 300:
        soup2 = _sopa()
        for tag in soup2(["script", "style", "noscript", "nav", "footer", "aside", "form", "iframe"]):
            tag.decompose()
        estreito = re.compile(r"relacionad|related|leia-?mais|leia-?tamb", re.I)
        for tag in soup2.find_all(attrs={"class": estreito}) + soup2.find_all(attrs={"id": estreito}):
            if not getattr(tag, "decomposed", False):
                tag.decompose()
        area2 = soup2.find("article") or soup2.find("main") or soup2.body or soup2
        texto = re.sub(r"\s+", " ", area2.get_text(separator=" ")).strip()

    return titulo, data_meta, texto


# Padrões (regex) para os números dos boletins. Cada item: (regex, divisor). O grupo 1 é o número.
# Truque de concordância de gênero do português para não confundir os dois campos mais ambíguos:
# "municípios afetados" (masc.) vs. "pessoas afetadas" (fem.) têm a MESMA palavra-raiz, então
# exigimos a terminação certa ("afetados" / "afetadas") em cada regex.
PADROES = {
    "municipios_afetados": [
        (re.compile(r"(\d{2,3})\s+(?:munic[íi]pios|cidades)[^.]{0,25}?afetados", re.I), 1),
        (re.compile(r"(?:soma|somam|chegou\s+a|subiu\s+para|total\s+de)\s+(\d{2,3})\s+munic[íi]pios", re.I), 1),
    ],
    "obitos": [
        (re.compile(r"(\d{1,3})\s+(?:mortes|mortos|[óo]bitos confirmados|[óo]bitos)", re.I), 1),
        (re.compile(r"(?:subiu|chegou)\s+(?:para|a)\s+(\d{1,3})\s+(?:o\s+n[úu]mero\s+de\s+)?(?:mortos|mortes)", re.I), 1),
    ],
    "desaparecidos": [
        (re.compile(r"(\d{1,3})\s+(?:pessoas\s+)?(?:seguem\s+|est[ãa]o\s+)?desaparecid", re.I), 1),
    ],
    "desalojados": [
        (re.compile(r"(\d{1,3}(?:\.\d{3})*)\s+(?:est[ãa]o\s+)?desalojad", re.I), 1),
    ],
    "pessoas_abrigos": [
        (re.compile(r"(\d{1,3}(?:\.\d{3})*)\s+(?:pessoas\s+)?(?:em|nos)\s+abrigos", re.I), 1),
    ],
    "pessoas_afetadas": [
        (re.compile(r"(\d{1,3}(?:\.\d{3})*)\s+(?:pessoas\s+)?(?:j[áa]\s+)?(?:foram\s+)?afetadas", re.I), 1),
    ],
    "nivel_guaiba_m": [
        (re.compile(r"gua[íi]ba[^.]{0,80}?(\d{1}[.,]\d{1,2})\s*m\b", re.I), 1),
        (re.compile(r"(\d{1}[.,]\d{1,2})\s*m[^.]{0,40}?gua[íi]ba", re.I), 1),
    ],
}

# Faixas plausíveis: valor fora da faixa é descartado (evita capturar números de outro contexto).
FAIXAS = {
    "municipios_afetados": (1, 497),
    "obitos": (1, 300),
    "desaparecidos": (1, 200),
    "desalojados": (100, 800_000),
    "pessoas_abrigos": (50, 150_000),
    "pessoas_afetadas": (1_000, 3_000_000),
    "nivel_guaiba_m": (1, 10),
}


def extrair_campo(campo, texto):
    '''Primeiro casamento válido (dentro da faixa). Devolve (valor, trecho_de_evidência).'''
    lo, hi = FAIXAS[campo]
    for regex, divisor in PADROES[campo]:
        for m in regex.finditer(texto):
            v = parse_num_ptbr(m.group(1))
            if v is None:
                continue
            v = v / divisor
            if lo <= v <= hi:
                return v, texto[max(0, m.start() - 60): m.end() + 60]
    return None, None


print("Extratores prontos.")


Extratores prontos.


In [ ]:
registros_scraping = []

for item in URLS_ENCHENTES:
    url = item["url"]
    dominio = urlparse(url).netloc
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", url.rstrip("/").split("/")[-1])[:80]
    caminho_bruto = os.path.join(DIR_BRUTOS_SCRAPING, f"{item['data_ref']}_{slug}.html")

    # Respeita robots.txt (Seção 2)
    if item["robots"] == "bloqueado" or (item["robots"] == "indeterminado" and not ROBOTS_INDETERMINADO_PROSSEGUIR):
        print(f"PULADO (robots={item['robots']}) {item['data_ref']} {dominio}")
        registrar_provenancia(
            fonte=item["fonte"], url=url, metodo="(não coletado)", parametros={},
            observacao=f"URL pulada: robots.txt = {item['robots']}",
        )
        continue

    try:
        resp = http_get(url, tentativas=2, timeout=20)
        resp.raise_for_status()

        # 1) PRESERVAÇÃO DO BRUTO: bytes exatamente como vieram, antes de qualquer parsing.
        sha = salvar_bruto(caminho_bruto, resp.content)
        registrar_provenancia(
            fonte=item["fonte"], url=url, metodo="GET (requests + BeautifulSoup)",
            parametros={"headers": "User-Agent customizado", "timeout": 20},
            observacao=f"HTTP {resp.status_code}", arquivo=caminho_bruto, sha256=sha,
        )

        # 2) Extração (BeautifulSoup recebe bytes e detecta o encoding sozinho)
        titulo, data_meta, texto = extrair_texto_pagina(resp.content)
        registro = {
            "data_ref": item["data_ref"], "fonte": item["fonte"], "url": url, "titulo": titulo,
            "data_publicacao_meta": data_meta,
            "data_divergente": bool(data_meta and data_meta[:7] != item["data_ref"][:7]),
            "arquivo_bruto": caminho_bruto,
        }
        evidencias = {}
        for campo in PADROES:
            valor, trecho = extrair_campo(campo, texto)
            registro[campo] = valor
            if trecho:
                evidencias[campo] = trecho
        registro["evidencias"] = json.dumps(evidencias, ensure_ascii=False)
        registros_scraping.append(registro)
        print(f"OK   {item['data_ref']}  {dominio}")

    except Exception as e:
        print(f"FALHOU {item['data_ref']} {dominio}: {e}")
        registrar_provenancia(
            fonte=item["fonte"], url=url, metodo="GET (requests)", parametros={}, observacao=f"ERRO: {e}",
        )

    time.sleep(max(DELAY_ENTRE_REQUISICOES, item.get("crawl_delay") or 0))

df_scraping_raw = pd.DataFrame(registros_scraping)
if len(df_scraping_raw):
    n_div = int(df_scraping_raw["data_divergente"].sum())
    if n_div:
        print(f"\n[atenção] {n_div} página(s) com data publicada em mês diferente de data_ref — confira `data_publicacao_meta`.")
df_scraping_raw


OK   2024-05-01  sul21.com.br
OK   2024-05-02  sul21.com.br
OK   2024-05-02  diariodonordeste.verdesmares.com.br
OK   2024-05-03  sul21.com.br
OK   2024-05-08  sul21.com.br
OK   2024-05-13  ihu.unisinos.br
OK   2024-05-16  sul21.com.br
OK   2024-05-17  sul21.com.br
OK   2024-05-17  www.correiobraziliense.com.br
FALHOU 2025-04-24 www.revistaoeste.com: 403 Client Error: Forbidden for url: https://www.revistaoeste.com/brasil/rio-grande-do-sul-confirma-184-vitima-das-enchentes-de-2024/


,data_ref,fonte,url,titulo,data_publicacao_meta,data_divergente,arquivo_bruto,municipios_afetados,obitos,desaparecidos,desalojados,pessoas_abrigos,pessoas_afetadas,nivel_guaiba_m,evidencias
0,2024-05-01,Sul21,https://sul21.com.br/?p=235018,Sobe para 10 o número de mortos em decorrência...,2024-05-01,False,./dados_brutos/scraping_enchentes/2024-05-01__...,104.0,NaN,21.0,NaN,NaN,NaN,None,"{""municipios_afetados"": ""mero de óbitos e 21 p..."
1,2024-05-02,Sul21,https://sul21.com.br/?p=235344,Rio Grande do Sul já tem 29 mortos e 60 desapa...,2024-05-02,False,./dados_brutos/scraping_enchentes/2024-05-02__...,154.0,29.0,60.0,10242.0,4645.0,71306.0,None,"{""municipios_afetados"": ""o ao vivo na noite de..."
2,2024-05-02,Diário do Nordeste,https://diariodonordeste.verdesmares.com.br/ul...,"Rio Grande do Sul tem 29 mortos em chuvas, atu...",2024-05-02,False,./dados_brutos/scraping_enchentes/2024-05-02_r...,NaN,NaN,60.0,10242.0,154.0,NaN,None,"{""desaparecidos"": ""Civil do Estado, divulgado ..."
3,2024-05-03,Sul21,https://sul21.com.br/?p=235669,Sobe para 39 o número de mortos em enchentes n...,2024-05-03,False,./dados_brutos/scraping_enchentes/2024-05-03__...,NaN,NaN,68.0,8168.0,NaN,NaN,None,"{""desaparecidos"": ""| 17:40 Sobe para 39 o núme..."
4,2024-05-08,Sul21,https://sul21.com.br/?p=237057,Chuvas no RS já causaram a morte de 100 pessoa...,2024-05-08,False,./dados_brutos/scraping_enchentes/2024-05-08__...,NaN,NaN,128.0,158992.0,66434.0,NaN,None,"{""desaparecidos"": ""4 | 10:27 Chuvas no RS já c..."
5,2024-05-13,IHU/Unisinos,https://ihu.unisinos.br/639521-extensao-territ...,Extensão territorial e número de afetados torn...,None,False,./dados_brutos/scraping_enchentes/2024-05-13_6...,497.0,148.0,NaN,NaN,NaN,NaN,None,"{""municipios_afetados"": ""cisou ser evacuado. S..."
6,2024-05-16,Sul21,https://sul21.com.br/?p=238281,Enchentes causam 151 mortes e deixam 615 mil p...,2024-05-16,False,./dados_brutos/scraping_enchentes/2024-05-16__...,NaN,151.0,104.0,NaN,NaN,NaN,None,"{""obitos"": ""Geral | 16 de maio de 2024 | 09:53..."
7,2024-05-17,Sul21,https://sul21.com.br/?p=238419,Vítimas fatais de enchentes chegam a 154 pesso...,2024-05-17,False,./dados_brutos/scraping_enchentes/2024-05-17__...,NaN,NaN,98.0,NaN,NaN,NaN,None,"{""desaparecidos"": ""2 Vítimas fatais de enchent..."
8,2024-05-17,Correio Braziliense,https://www.correiobraziliense.com.br/brasil/2...,Mais de 90% dos municípios gaúchos são atingid...,2024-05-17,False,./dados_brutos/scraping_enchentes/2024-05-17_6...,NaN,NaN,NaN,540192.0,NaN,NaN,None,"{""desalojados"": ""odo, 806 pessoas ficaram feri..."


### Conferência manual (prioridade sobre o automático)

Como a extração automática depende de regex sobre texto jornalístico (formato não padronizado),
é normal que alguns campos venham vazios — nem toda matéria cita todos os números — ou errados.
A tabela abaixo é uma **conferência manual** das mesmas páginas de `URLS_ENCHENTES` (e, para
13/05, de uma citação de boletim oficial dentro da matéria do IHU). Na Seção 6.1 ela tem
**prioridade** sobre o valor automático, e toda divergência é listada.


In [ ]:
dados_manuais_conferencia = [
    {"data_ref": "2024-05-01", "municipios_afetados": 104, "obitos": 10,  "desaparecidos": 21,  "desalojados": 1431,   "pessoas_abrigos": 1145,  "pessoas_afetadas": None,    "nivel_guaiba_m": None},
    {"data_ref": "2024-05-02", "municipios_afetados": 154, "obitos": 29,  "desaparecidos": 60,  "desalojados": 10242,  "pessoas_abrigos": 4645,  "pessoas_afetadas": 71306,   "nivel_guaiba_m": None},
    {"data_ref": "2024-05-03", "municipios_afetados": 265, "obitos": 39,  "desaparecidos": 68,  "desalojados": 24080,  "pessoas_abrigos": 8168,  "pessoas_afetadas": 351639,  "nivel_guaiba_m": None},
    {"data_ref": "2024-05-08", "municipios_afetados": 414, "obitos": 100, "desaparecidos": 128, "desalojados": 158992, "pessoas_abrigos": 66434, "pessoas_afetadas": None,    "nivel_guaiba_m": None},
    {"data_ref": "2024-05-13", "municipios_afetados": 450, "obitos": 148, "desaparecidos": 124, "desalojados": 538000, "pessoas_abrigos": None,  "pessoas_afetadas": 2100000, "nivel_guaiba_m": 5.21},
    {"data_ref": "2024-05-16", "municipios_afetados": None,"obitos": 151, "desaparecidos": 104, "desalojados": None,   "pessoas_abrigos": None,  "pessoas_afetadas": None,    "nivel_guaiba_m": None},
    {"data_ref": "2024-05-17", "municipios_afetados": 461, "obitos": 154, "desaparecidos": 98,  "desalojados": 540192, "pessoas_abrigos": 77199, "pessoas_afetadas": 2281830, "nivel_guaiba_m": None},
    {"data_ref": "2025-04-24", "municipios_afetados": 478, "obitos": 184, "desaparecidos": 25,  "desalojados": None,   "pessoas_abrigos": None,  "pessoas_afetadas": None,    "nivel_guaiba_m": None},
]
df_manual = pd.DataFrame(dados_manuais_conferencia)
for c in ["municipios_afetados", "obitos", "desaparecidos", "desalojados", "pessoas_abrigos", "pessoas_afetadas", "nivel_guaiba_m"]:
    df_manual[c] = df_manual[c].astype(float)

caminho_manual = os.path.join(DIR_BRUTOS_SCRAPING, "conferencia_manual_boletins.csv")
df_manual.to_csv(caminho_manual, index=False)
with open(caminho_manual, "rb") as f:
    _sha_manual = sha256_bytes(f.read())
registrar_provenancia(
    fonte="Conferência manual (mesmas URLs de URLS_ENCHENTES, e IHU citando o boletim de 13/5)",
    url="ver coluna url em URLS_ENCHENTES",
    metodo="Leitura manual pelos autores do trabalho",
    parametros={}, observacao="valores digitados pelos autores; revisar contra as páginas",
    arquivo=caminho_manual, sha256=_sha_manual,
)
df_manual


,data_ref,municipios_afetados,obitos,desaparecidos,desalojados,pessoas_abrigos,pessoas_afetadas,nivel_guaiba_m
0,2024-05-01,104.0,10.0,21.0,1431.0,1145.0,NaN,NaN
1,2024-05-02,154.0,29.0,60.0,10242.0,4645.0,71306.0,NaN
2,2024-05-03,265.0,39.0,68.0,24080.0,8168.0,351639.0,NaN
3,2024-05-08,414.0,100.0,128.0,158992.0,66434.0,NaN,NaN
4,2024-05-13,450.0,148.0,124.0,538000.0,NaN,2100000.0,5.21
5,2024-05-16,NaN,151.0,104.0,NaN,NaN,NaN,NaN
6,2024-05-17,461.0,154.0,98.0,540192.0,77199.0,2281830.0,NaN
7,2025-04-24,478.0,184.0,25.0,NaN,NaN,NaN,NaN


## 4. Fonte 2 — API do IBGE

Duas famílias de chamadas:

1. **Localidades** — lista oficial dos 497 municípios do Rio Grande do Sul (código IBGE e nome). É
   uma dimensão de apoio para as fases futuras (agrupamento/geolocalização). *Não* traz população.
2. **SIDRA/Agregados** — indicadores econômicos: **IPCA mensal** (tabela 7060) e **taxa de
   desocupação** da PNAD Contínua (tabela 4099, trimestral, nível UF).

**Como os parâmetros são definidos:**

- **IDs de tabela fixos** (Seção 1), conferidos em tempo de execução pelo nome e pelos metadados
  impressos abaixo — em vez de pegar o "primeiro resultado" de uma busca no catálogo, cuja ordem é arbitrária.
- **Localidade descoberta pelo nome** via `/agregados/{id}/localidades/{nivel}` (nada de código
  hardcoded): para o IPCA procura Porto Alegre diretamente (é área de coleta histórica do índice) e,
  se por algum motivo não existir na tabela, usa um proxy (Região Sul, depois Brasil); para a
  desocupação procura o Rio Grande do Sul (UF) e, se faltar, a Região Sul.
- **Períodos explícitos**: lista os períodos reais da tabela (`/agregados/{id}/periodos`) e
  seleciona os que intersectam a janela `JANELA_INICIO`–`JANELA_FIM`. A frequência (mensal ou
  trimestral) é inferida dos próprios códigos de período.


In [ ]:
URL_MUNICIPIOS_RS = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/RS/municipios"

resp = http_get(URL_MUNICIPIOS_RS, timeout=20)
resp.raise_for_status()
municipios_rs_raw = resp.json()

caminho_bruto_municipios = os.path.join(DIR_BRUTOS_IBGE, "municipios_rs_raw.json")
sha = salvar_json_bruto(caminho_bruto_municipios, municipios_rs_raw)
registrar_provenancia(
    fonte="API IBGE - Localidades", url=URL_MUNICIPIOS_RS, metodo="GET", parametros={},
    observacao=f"{len(municipios_rs_raw)} municípios", arquivo=caminho_bruto_municipios, sha256=sha,
)

if len(municipios_rs_raw) != 497:
    print(f"[aviso] esperados 497 municípios do RS, vieram {len(municipios_rs_raw)}.")
print(f"{len(municipios_rs_raw)} municípios do RS coletados.")
pd.json_normalize(municipios_rs_raw)[["id", "nome"]].head()


497 municípios do RS coletados.


,id,nome
0,4300034,Aceguá
1,4300059,Água Santa
2,4300109,Agudo
3,4300208,Ajuricaba
4,4300307,Alecrim


In [ ]:
BASE_API_AGREGADOS = "https://servicodados.ibge.gov.br/api/v3/agregados"
_MISSING_SIDRA = {"", "-", "..", "...", "X"}


def obter_json(url, fonte, parametros=None, caminho_bruto=None):
    '''GET + JSON + proveniência. Em caso de erro imprime o corpo da resposta (a API do IBGE
    costuma explicar o que está errado) e devolve None.'''
    try:
        resp = http_get(url, timeout=30)
    except Exception as e:
        print(f"[ERRO de rede] {fonte}: {e}")
        registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros, observacao=f"ERRO: {e}")
        return None
    if resp.status_code != 200:
        print(f"[ERRO HTTP {resp.status_code}] {fonte}\n   corpo: {resp.text[:400]}")
        registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros,
                              observacao=f"ERRO HTTP {resp.status_code}: {resp.text[:200]}")
        return None
    try:
        dados = resp.json()
    except ValueError:
        print(f"[ERRO] {fonte}: resposta não é JSON: {resp.text[:200]}")
        registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros, observacao="ERRO: resposta não-JSON")
        return None
    sha = salvar_json_bruto(caminho_bruto, dados) if caminho_bruto else None
    registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros,
                          observacao="OK", arquivo=caminho_bruto, sha256=sha)
    return dados


def niveis_da_tabela(meta):
    '''Conjunto de níveis territoriais (N1, N2, N3, N6, N7...) disponíveis na tabela.
    Aceita tanto {"Administrativo": ["N1","N3"], ...} quanto {"N3": true, ...}.'''
    nt = meta.get("nivelTerritorial", {})
    niveis = set()
    if isinstance(nt, dict):
        for k, v in nt.items():
            if isinstance(v, (list, tuple, set)):
                niveis.update(v)
            elif v and re.fullmatch(r"N\d+", str(k)):
                niveis.add(str(k))
    return niveis


def escolher_variavel(variaveis, palavras_chave):
    '''Escolhe a variável cujo nome contém alguma palavra-chave; senão, a primeira (com aviso).'''
    for v in variaveis:
        if any(p in v["nome"].lower() for p in palavras_chave):
            return v["id"]
    print(f"[aviso] nenhuma variável casou com {palavras_chave}; usando a primeira: {variaveis[0]['nome']}")
    return variaveis[0]["id"]


def montar_classificacao(meta, preferencias=("índice geral", "total", "geral")):
    '''Muitas tabelas exigem `classificacao`. Para cada classificação escolhe a categoria mais
    agregada (por nome, na ordem de `preferencias`); se nenhuma casar, usa a primeira.'''
    partes = []
    for classif in meta.get("classificacoes", []):
        cats = classif.get("categorias", [])
        if not cats:
            continue
        escolhida = None
        for pref in preferencias:
            escolhida = next((c for c in cats if pref in c["nome"].lower()), None)
            if escolhida:
                break
        escolhida = escolhida or cats[0]
        partes.append(f"{classif['id']}[{escolhida['id']}]")
    return "|".join(partes) if partes else None


def localizar_localidade(id_tabela, preferencias, niveis_disponiveis):
    '''preferencias = [(nivel, trecho_do_nome), ...] em ordem de preferência.
    Consulta /agregados/{id}/localidades/{nivel} e devolve (nivel, id, nome) do 1º casamento.'''
    cache = {}
    for nivel, trecho in preferencias:
        if nivel not in niveis_disponiveis:
            continue
        if nivel not in cache:
            cache[nivel] = obter_json(
                f"{BASE_API_AGREGADOS}/{id_tabela}/localidades/{nivel}",
                fonte=f"API IBGE - Localidades do agregado {id_tabela} ({nivel})",
                parametros={"nivel": nivel},
            ) or []
        for loc in cache[nivel]:
            if trecho in unidecode(loc["nome"]).lower():
                return nivel, str(loc["id"]), loc["nome"]
    return None


def inferir_frequencia(ids_periodos):
    ''''trimestral' se os códigos AAAAQQ só usam QQ = 01..04; 'mensal' caso contrário
    (inclui trimestre móvel, em que QQ é o mês final, 01..12).'''
    sufixos = [int(p[4:]) for p in ids_periodos if len(p) == 6 and p.isdigit()]
    return "trimestral" if sufixos and max(sufixos) <= 4 else "mensal"


def periodo_para_meses(codigo, frequencia):
    '''Converte um código de período do SIDRA em lista de 'AAAA-MM'.
    trimestral: 202304 (4º tri) -> ['2023-10','2023-11','2023-12'] | mensal: 202304 -> ['2023-04'].'''
    codigo = str(codigo)
    if len(codigo) != 6 or not codigo.isdigit():
        return []
    ano, suf = int(codigo[:4]), int(codigo[4:])
    if frequencia == "trimestral":
        return [f"{ano}-{m:02d}" for m in range(3 * suf - 2, 3 * suf + 1)] if 1 <= suf <= 4 else []
    return [f"{ano}-{suf:02d}"] if 1 <= suf <= 12 else []


def periodos_na_janela(id_tabela):
    '''Lista os períodos reais da tabela e devolve (ids_na_janela, frequencia).'''
    lista = obter_json(f"{BASE_API_AGREGADOS}/{id_tabela}/periodos",
                       fonte=f"API IBGE - Períodos do agregado {id_tabela}")
    if not lista:
        return [], None
    ids = [str(p["id"]) for p in lista]
    freq = inferir_frequencia(ids)
    janela = set(MESES_JANELA)
    return [i for i in ids if set(periodo_para_meses(i, freq)) & janela], freq


def montar_url_sidra(id_tabela, variavel, periodos, localidades, classificacao=None):
    url = (f"{BASE_API_AGREGADOS}/{id_tabela}/periodos/{'|'.join(periodos)}"
           f"/variaveis/{variavel}?localidades={localidades}")
    if classificacao:
        url += f"&classificacao={classificacao}"
    return url


def converter_valor_sidra(valor):
    s = str(valor).strip()
    if s in _MISSING_SIDRA:
        return None
    try:
        return float(s.replace(",", "."))
    except ValueError:
        return None


def sidra_para_df(sidra_json, nome_valor, frequencia):
    '''Achata o JSON do SIDRA (lista de variáveis -> 'resultados' -> 'series' -> 'serie') em um
    DataFrame tidy [ano_mes, periodo_original, localidade, <nome_valor>]. Trimestres viram 3 meses.'''
    colunas = ["ano_mes", "periodo_original", "localidade", nome_valor]
    linhas = []
    for variavel in sidra_json:
        for resultado in variavel.get("resultados", []):
            for serie in resultado.get("series", []):
                localidade = serie.get("localidade", {}).get("nome")
                for periodo, valor in serie.get("serie", {}).items():
                    v = converter_valor_sidra(valor)
                    for ano_mes in periodo_para_meses(periodo, frequencia):
                        linhas.append({"ano_mes": ano_mes, "periodo_original": str(periodo),
                                       "localidade": localidade, nome_valor: v})
    df = pd.DataFrame(linhas, columns=colunas)
    df[nome_valor] = df[nome_valor].astype(float)
    return df.drop_duplicates(subset="ano_mes", keep="last").reset_index(drop=True)


def coletar_indicador(id_tabela, rotulo, preferencias_localidade, palavras_variavel, arquivo_bruto):
    '''Pipeline completo para um agregado: metadados -> variável -> classificação -> localidade ->
    períodos -> dados. Devolve dict com o JSON bruto e os parâmetros usados, ou None se falhar.'''
    meta = obter_json(f"{BASE_API_AGREGADOS}/{id_tabela}/metadados",
                      fonte=f"API IBGE - Metadados do agregado {id_tabela}")
    if meta is None:
        return None

    print(f"[{rotulo}] Tabela {id_tabela}: {meta.get('nome')}")
    print("   Periodicidade:", meta.get("periodicidade"))
    print("   Variáveis:")
    for v in meta.get("variaveis", []):
        print("     ", v["id"], "-", v["nome"])
    niveis = niveis_da_tabela(meta)
    print("   Níveis territoriais:", sorted(niveis))

    variavel = escolher_variavel(meta["variaveis"], palavras_variavel)
    classificacao = montar_classificacao(meta)
    achado = localizar_localidade(id_tabela, preferencias_localidade, niveis)
    if achado is None:
        print(f"[{rotulo}] Nenhuma das localidades preferidas existe nesta tabela: {preferencias_localidade}")
        return None
    nivel, loc_id, loc_nome = achado
    periodos, frequencia = periodos_na_janela(id_tabela)
    if not periodos:
        print(f"[{rotulo}] Nenhum período da tabela intersecta a janela {JANELA_INICIO}..{JANELA_FIM}.")
        return None

    url = montar_url_sidra(id_tabela, variavel, periodos, f"{nivel}[{loc_id}]", classificacao)
    print(f"   Localidade: {loc_nome} ({nivel}[{loc_id}]) | frequência: {frequencia} | "
          f"{len(periodos)} períodos | classificação: {classificacao}")
    dados = obter_json(
        url, fonte=f"API IBGE - SIDRA ({rotulo})",
        parametros={"agregado": id_tabela, "variavel": variavel, "localidades": f"{nivel}[{loc_id}]",
                    "classificacao": classificacao, "periodos": f"{periodos[0]}..{periodos[-1]}"},
        caminho_bruto=arquivo_bruto,
    )
    if dados is None:
        return None
    return {"json": dados, "frequencia": frequencia, "localidade_nome": loc_nome, "url": url,
            "tabela_nome": meta.get("nome")}


print("Funções da API prontas.")


Funções da API prontas.


In [ ]:
# (Opcional, só para conferência) O ID pinado aparece no catálogo público do SIDRA? Com que nome?
catalogo = obter_json(BASE_API_AGREGADOS, fonte="API IBGE - Catálogo de Agregados (SIDRA)")
if catalogo:
    nomes_por_id = {str(a["id"]): a["nome"] for grupo in catalogo for a in grupo.get("agregados", [])}
    for rotulo, tid in [("IPCA", ID_TABELA_IPCA), ("Desocupação", ID_TABELA_DESOCUPACAO)]:
        print(f"{rotulo}: tabela {tid} -> {nomes_por_id.get(str(tid), '!! NÃO ENCONTRADA no catálogo !!')}")
else:
    print("Catálogo indisponível; a conferência será feita pelos metadados nas próximas células.")


IPCA: tabela 7060 -> IPCA - Variação mensal, acumulada no ano, acumulada em 12 meses e peso mensal, para o índice geral, grupos, subgrupos, itens e subitens de produtos e serviços (a partir de janeiro/2020)
Desocupação: tabela 4099 -> Taxas de desocupação e de subutilização da força de trabalho, na semana de referência, das pessoas de 14 anos ou mais de idade


In [ ]:
# --- IPCA: Porto Alegre é área de coleta histórica do índice -> procura direto; ------------------
# --- se por algum motivo não existir na tabela, cai para Região Sul e depois Brasil. -------------
res_ipca = coletar_indicador(
    ID_TABELA_IPCA, "IPCA",
    preferencias_localidade=[("N6", "porto alegre"), ("N7", "porto alegre"),  # município / RM
                             ("N2", "sul"), ("N1", "brasil")],                 # proxies de último recurso
    palavras_variavel=["variação mensal"],
    arquivo_bruto=os.path.join(DIR_BRUTOS_IBGE, "ipca_raw.json"),
)
if res_ipca is None:
    print("\nIPCA NÃO coletado. Veja a mensagem de erro acima (corpo da resposta da API) e ajuste ID_TABELA_IPCA.")
elif "porto alegre" not in unidecode(res_ipca["localidade_nome"]).lower():
    print(f"\n[ATENÇÃO] Porto Alegre não existe na tabela {ID_TABELA_IPCA} (inesperado — confira a tabela). "
          f"Usando PROXY: '{res_ipca['localidade_nome']}'. Registre isso no dataset card (Seção 10, A.5).")


[IPCA] Tabela 7060: IPCA - Variação mensal, acumulada no ano, acumulada em 12 meses e peso mensal, para o índice geral, grupos, subgrupos, itens e subitens de produtos e serviços (a partir de janeiro/2020)
   Periodicidade: {'frequencia': 'mensal', 'inicio': 202001, 'fim': 202608}
   Variáveis:
      63 - IPCA - Variação mensal
      69 - IPCA - Variação acumulada no ano
      2265 - IPCA - Variação acumulada em 12 meses
      66 - IPCA - Peso mensal
   Níveis territoriais: ['N1', 'N6', 'N7']
   Localidade: Brasil (N1[1]) | frequência: mensal | 25 períodos | classificação: 315[7169]

[ATENÇÃO] Porto Alegre não existe na tabela 7060 (inesperado — confira a tabela). Usando PROXY: 'Brasil'. Registre isso no dataset card (Seção 10, A.5).


In [ ]:
# --- Taxa de desocupação (PNAD Contínua): Rio Grande do Sul (UF); se faltar, Região Sul; senão Brasil ---
res_desoc = coletar_indicador(
    ID_TABELA_DESOCUPACAO, "Desocupação",
    preferencias_localidade=[("N3", "rio grande do sul"), ("N2", "sul"), ("N1", "brasil")],
    palavras_variavel=["taxa de desocupação", "desocupação"],
    arquivo_bruto=os.path.join(DIR_BRUTOS_IBGE, "desocupacao_raw.json"),
)
if res_desoc is None:
    print("\nDesocupação NÃO coletada. Veja a mensagem de erro acima e confira ID_TABELA_DESOCUPACAO.")
else:
    if res_desoc["frequencia"] == "trimestral":
        print("\n[nota] Série TRIMESTRAL: cada trimestre será repetido nos seus 3 meses (Seção 6.2).")
    if "rio grande do sul" not in unidecode(res_desoc["localidade_nome"]).lower():
        print(f"[ATENÇÃO] Usando '{res_desoc['localidade_nome']}' no lugar do Rio Grande do Sul. Registre no dataset card.")


[Desocupação] Tabela 4099: Taxas de desocupação e de subutilização da força de trabalho, na semana de referência, das pessoas de 14 anos ou mais de idade
   Periodicidade: {'frequencia': 'trimestral', 'inicio': 201201, 'fim': 202602}
   Variáveis:
      4099 - Taxa de desocupação, na semana de referência, das pessoas de 14 anos ou mais de idade
      4103 - Coeficiente de variação - Taxa de desocupação, na semana de referência, das pessoas de 14 anos ou mais de idade
      4114 - Taxa combinada de desocupação e de subocupação por insuficiência de horas trabalhadas, na semana de referência, das pessoas de 14 anos ou mais de idade
      4115 - Coeficiente de variação - Taxa combinada de desocupação e de subocupação por insuficiência de horas trabalhadas, na semana de referência, das pessoas de 14 anos ou mais de idade
      4116 - Taxa combinada de desocupação e força de trabalho potencial, na semana de referência, das pessoas de 14 anos ou mais de idade
      4117 - Coeficiente de varia

## 5. Conferência da preservação do dado bruto (item 3.3)

In [ ]:
for pasta in [DIR_BRUTOS_SCRAPING, DIR_BRUTOS_IBGE]:
    print(f"\n{pasta}/")
    for arq in sorted(os.listdir(pasta)):
        caminho = os.path.join(pasta, arq)
        with open(caminho, "rb") as f:
            h = sha256_bytes(f.read())[:12]
        print(f"  - {arq}  ({os.path.getsize(caminho) / 1024:.1f} KB, sha256 {h}…)")



./dados_brutos/scraping_enchentes/
  - 2024-05-01__p_235018.html  (132.5 KB, sha256 9d8fcbd5727d…)
  - 2024-05-02__p_235344.html  (141.7 KB, sha256 2471cca2e2c7…)
  - 2024-05-02_rio_grande_do_sul_tem_29_mortos_em_chuvas_atualiza_defesa_civil_1_3507808.html  (139.3 KB, sha256 359101a7e0cf…)
  - 2024-05-03__p_235669.html  (136.9 KB, sha256 0350930475f8…)
  - 2024-05-08__p_237057.html  (135.7 KB, sha256 860feedb82e1…)
  - 2024-05-13_639521_extensao_territorial_e_numero_de_afetados_tornam_tragedia_no_rs_inedita_n.html  (63.9 KB, sha256 0ade60ab509f…)
  - 2024-05-16__p_238281.html  (135.6 KB, sha256 abe7492eff70…)
  - 2024-05-17_6859198_mais_de_90_dos_municipios_gauchos_sao_atingidos_pela_tragedia_climatica_.html  (185.9 KB, sha256 14fc6b6276e3…)
  - 2024-05-17__p_238419.html  (135.6 KB, sha256 31ce3bec3e91…)
  - conferencia_manual_boletins.csv  (0.5 KB, sha256 fa1d7e80bafa…)

./dados_brutos/api_ibge/
  - desocupacao_raw.json  (0.8 KB, sha256 2441580d12a7…)
  - ipca_raw.json  (1.4 KB, sha2

## 6. Tratamento, limpeza e integração das fontes

### 6.1 Série das enchentes (scraping + conferência manual)
Une o resultado automático (`df_scraping_raw`) com a conferência manual (`df_manual`) pela `data_ref`.
Para cada campo, **o valor manual tem prioridade** e a coluna `origem_<campo>` registra de onde veio
(`manual`, `automatico` ou vazio). Divergências entre os dois são listadas em `df_divergencias`
para o grupo decidir. O resultado é a base **por boletim** (`df_boletins`).

### 6.2 Indicadores econômicos (API)
O JSON do SIDRA é achatado por `sidra_para_df` (Seção 4). Séries **trimestrais** são expandidas para
os 3 meses do trimestre (o valor se repete), preservando o código original em `periodo_original`.

### 6.3 Integração (base mensal)
Chave de integração = `ano_mes`. Os boletins são **agregados por mês** (evita repetir o mesmo IPCA
em várias linhas e inflar artificialmente o n) e juntados a um **calendário completo** da janela de
análise. Assim, meses **sem boletim** continuam na base (com campos da enchente vazios), mantendo o
contexto econômico de antes/depois do desastre. Agregações por mês: todos os campos numéricos
(`municipios_afetados`, `obitos`, `desaparecidos`, `desalojados`, `pessoas_abrigos`,
`pessoas_afetadas`, `nivel_guaiba_m`) → **máximo** (o pior ponto do mês; diferente do caso da
estiagem, aqui valores maiores sempre significam pior situação, inclusive para o nível do rio);
`n_boletins` → contagem.

> **Rótulo `em_crise`:** `True` se `municipios_afetados >= LIMIAR_CRISE_MUNICIPIOS`; `False` se abaixo;
> **vazio (`<NA>`) quando não há boletim no mês** (ausência de dado ≠ ausência de crise).
> O limiar é arbitrário. Como `em_crise` é função direta de `municipios_afetados`, **não use as duas
> juntas** em modelos futuros (vazamento de alvo).

Nomes de município (quando usados) são normalizados com `unidecode` + `str.upper().strip()`.


In [ ]:
# --- 6.1 Série das enchentes: manual tem prioridade; registra origem e divergências -------------
CAMPOS = ["municipios_afetados", "obitos", "desaparecidos", "desalojados", "pessoas_abrigos",
          "pessoas_afetadas", "nivel_guaiba_m"]
COLS_AUTO = ["data_ref", "fonte", "url", "titulo", "data_publicacao_meta", "data_divergente",
             "arquivo_bruto", "evidencias"] + CAMPOS

df_auto = df_scraping_raw.copy() if len(df_scraping_raw) else pd.DataFrame(columns=COLS_AUTO)
df_est = df_auto.merge(df_manual, on="data_ref", how="outer", suffixes=("_auto", "_manual"))

divergencias = []
for c in CAMPOS:
    manual, auto = df_est[f"{c}_manual"], df_est[f"{c}_auto"]
    df_est[c] = pd.to_numeric(manual.combine_first(auto), errors="coerce")   # manual tem prioridade
    df_est[f"origem_{c}"] = np.where(manual.notna(), "manual", np.where(auto.notna(), "automatico", None))
    dif = df_est[manual.notna() & auto.notna() & (manual != auto)]
    for _, r in dif.iterrows():
        divergencias.append({"data_ref": r["data_ref"], "fonte": r.get("fonte"), "campo": c,
                             "automatico": r[f"{c}_auto"], "manual": r[f"{c}_manual"]})

df_divergencias = pd.DataFrame(divergencias, columns=["data_ref", "fonte", "campo", "automatico", "manual"])
if len(df_divergencias):
    print(f"[atenção] {len(df_divergencias)} divergência(s) entre extração automática e conferência manual "
          "(vale o manual; confira qual está certo na página original):")
    display(df_divergencias)
else:
    print("Sem divergências entre automático e manual nos campos em que ambos existem.")

df_est["data_ref"] = pd.to_datetime(df_est["data_ref"])
df_est["ano_mes"] = df_est["data_ref"].dt.strftime("%Y-%m")

COLS_BOLETIM = (["data_ref", "ano_mes", "fonte", "url"] + CAMPOS + [f"origem_{c}" for c in CAMPOS]
                + ["data_publicacao_meta", "data_divergente"])
df_boletins = (df_est.reindex(columns=COLS_BOLETIM)
                     .sort_values(["data_ref", "fonte"]).reset_index(drop=True))
df_boletins


[atenção] 3 divergência(s) entre extração automática e conferência manual (vale o manual; confira qual está certo na página original):


,data_ref,fonte,campo,automatico,manual
0,2024-05-13,IHU/Unisinos,municipios_afetados,497.0,450.0
1,2024-05-03,Sul21,desalojados,8168.0,24080.0
2,2024-05-02,Diário do Nordeste,pessoas_abrigos,154.0,4645.0


,data_ref,ano_mes,fonte,url,municipios_afetados,obitos,desaparecidos,desalojados,pessoas_abrigos,pessoas_afetadas,nivel_guaiba_m,origem_municipios_afetados,origem_obitos,origem_desaparecidos,origem_desalojados,origem_pessoas_abrigos,origem_pessoas_afetadas,origem_nivel_guaiba_m,data_publicacao_meta,data_divergente
0,2024-05-01,2024-05,Sul21,https://sul21.com.br/?p=235018,104.0,10.0,21.0,1431.0,1145.0,NaN,NaN,manual,manual,manual,manual,manual,None,None,2024-05-01,False
1,2024-05-02,2024-05,Diário do Nordeste,https://diariodonordeste.verdesmares.com.br/ul...,154.0,29.0,60.0,10242.0,4645.0,71306.0,NaN,manual,manual,manual,manual,manual,manual,None,2024-05-02,False
2,2024-05-02,2024-05,Sul21,https://sul21.com.br/?p=235344,154.0,29.0,60.0,10242.0,4645.0,71306.0,NaN,manual,manual,manual,manual,manual,manual,None,2024-05-02,False
3,2024-05-03,2024-05,Sul21,https://sul21.com.br/?p=235669,265.0,39.0,68.0,24080.0,8168.0,351639.0,NaN,manual,manual,manual,manual,manual,manual,None,2024-05-03,False
4,2024-05-08,2024-05,Sul21,https://sul21.com.br/?p=237057,414.0,100.0,128.0,158992.0,66434.0,NaN,NaN,manual,manual,manual,manual,manual,None,None,2024-05-08,False
5,2024-05-13,2024-05,IHU/Unisinos,https://ihu.unisinos.br/639521-extensao-territ...,450.0,148.0,124.0,538000.0,NaN,2100000.0,5.21,manual,manual,manual,manual,None,manual,manual,None,False
6,2024-05-16,2024-05,Sul21,https://sul21.com.br/?p=238281,NaN,151.0,104.0,NaN,NaN,NaN,NaN,None,manual,manual,None,None,None,None,2024-05-16,False
7,2024-05-17,2024-05,Correio Braziliense,https://www.correiobraziliense.com.br/brasil/2...,461.0,154.0,98.0,540192.0,77199.0,2281830.0,NaN,manual,manual,manual,manual,manual,manual,None,2024-05-17,False
8,2024-05-17,2024-05,Sul21,https://sul21.com.br/?p=238419,461.0,154.0,98.0,540192.0,77199.0,2281830.0,NaN,manual,manual,manual,manual,manual,manual,None,2024-05-17,False
9,2025-04-24,2025-04,NaN,NaN,478.0,184.0,25.0,NaN,NaN,NaN,NaN,manual,manual,manual,None,None,None,None,NaN,NaN


In [ ]:
# --- 6.2 Indicadores do IBGE em DataFrames tidy: ano_mes + valor -------------------------------
def _df_indicador(res, nome_valor, prefixo):
    colunas = ["ano_mes", nome_valor, f"{prefixo}_localidade", f"{prefixo}_periodo_original"]
    if res is None:
        print(f"[aviso] {nome_valor} não foi coletado na Seção 4 — seguindo com DataFrame vazio "
              "(a coluna ficará toda vazia na base).")
        return pd.DataFrame(columns=colunas)
    df = sidra_para_df(res["json"], nome_valor, res["frequencia"])
    return df.rename(columns={"localidade": f"{prefixo}_localidade",
                              "periodo_original": f"{prefixo}_periodo_original"})[colunas]


df_ipca = _df_indicador(res_ipca if "res_ipca" in globals() else None, "ipca_variacao_mensal", "ipca")
df_desoc = _df_indicador(res_desoc if "res_desoc" in globals() else None, "taxa_desocupacao", "desocupacao")

print("IPCA:", df_ipca.shape, "| Desocupação:", df_desoc.shape)
print("\nIPCA (amostra):")
display(df_ipca.head())
print("\nDesocupação (amostra — confira se cada trimestre aparece repetido nos 3 meses):")
display(df_desoc.head(6))


IPCA: (25, 4) | Desocupação: (27, 4)

IPCA (amostra):


,ano_mes,ipca_variacao_mensal,ipca_localidade,ipca_periodo_original
0,2023-06,-0.08,Brasil,202306
1,2023-07,0.12,Brasil,202307
2,2023-08,0.23,Brasil,202308
3,2023-09,0.26,Brasil,202309
4,2023-10,0.24,Brasil,202310



Desocupação (amostra — confira se cada trimestre aparece repetido nos 3 meses):


,ano_mes,taxa_desocupacao,desocupacao_localidade,desocupacao_periodo_original
0,2023-04,5.3,Rio Grande do Sul,202302
1,2023-05,5.3,Rio Grande do Sul,202302
2,2023-06,5.3,Rio Grande do Sul,202302
3,2023-07,5.4,Rio Grande do Sul,202303
4,2023-08,5.4,Rio Grande do Sul,202303
5,2023-09,5.4,Rio Grande do Sul,202303


In [ ]:
# --- 6.3 Integração final (base mensal): calendário da janela + enchentes + IPCA + desocupação ----
agg_boletins = (df_boletins.groupby("ano_mes")
                .agg(n_boletins=("data_ref", "size"),
                     municipios_afetados=("municipios_afetados", "max"),
                     obitos=("obitos", "max"),
                     desaparecidos=("desaparecidos", "max"),
                     desalojados=("desalojados", "max"),
                     pessoas_abrigos=("pessoas_abrigos", "max"),
                     pessoas_afetadas=("pessoas_afetadas", "max"),
                     nivel_guaiba_m=("nivel_guaiba_m", "max"))
                .reset_index())

fora_da_janela = sorted(set(agg_boletins["ano_mes"]) - set(MESES_JANELA))
if fora_da_janela:
    print(f"[aviso] boletins em meses fora da janela (serão descartados da base mensal): {fora_da_janela}")

calendario = pd.DataFrame({"ano_mes": MESES_JANELA})
base_mensal = (calendario
               .merge(agg_boletins, on="ano_mes", how="left")
               .merge(df_ipca, on="ano_mes", how="left")
               .merge(df_desoc, on="ano_mes", how="left"))
base_mensal["n_boletins"] = base_mensal["n_boletins"].fillna(0).astype(int)

# Rótulo com dado ausente preservado (NÃO transformar NaN em "sem crise")
_m = base_mensal["municipios_afetados"]
base_mensal["em_crise"] = (_m >= LIMIAR_CRISE_MUNICIPIOS).astype("boolean").mask(_m.isna())

# Dimensão de municípios do RS (código, nome, nome normalizado) — apoio para fases futuras.
# Ainda NÃO entra no merge: nenhuma das fontes atuais tem dado em nível de município.
df_municipios_rs = pd.json_normalize(municipios_rs_raw)[["id", "nome"]].rename(
    columns={"id": "municipio_id", "nome": "municipio_nome"})
df_municipios_rs["municipio_nome_normalizado"] = df_municipios_rs["municipio_nome"].apply(
    lambda s: unidecode(s).upper().strip())

base_mensal


,ano_mes,n_boletins,municipios_afetados,obitos,desaparecidos,desalojados,pessoas_abrigos,pessoas_afetadas,nivel_guaiba_m,ipca_variacao_mensal,ipca_localidade,ipca_periodo_original,taxa_desocupacao,desocupacao_localidade,desocupacao_periodo_original,em_crise
0,2023-06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.08,Brasil,202306,5.3,Rio Grande do Sul,202302,<NA>
1,2023-07,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.12,Brasil,202307,5.4,Rio Grande do Sul,202303,<NA>
2,2023-08,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.23,Brasil,202308,5.4,Rio Grande do Sul,202303,<NA>
3,2023-09,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.26,Brasil,202309,5.4,Rio Grande do Sul,202303,<NA>
4,2023-10,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.24,Brasil,202310,5.2,Rio Grande do Sul,202304,<NA>
5,2023-11,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.28,Brasil,202311,5.2,Rio Grande do Sul,202304,<NA>
6,2023-12,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.56,Brasil,202312,5.2,Rio Grande do Sul,202304,<NA>
7,2024-01,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.42,Brasil,202401,5.8,Rio Grande do Sul,202401,<NA>
8,2024-02,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.83,Brasil,202402,5.8,Rio Grande do Sul,202401,<NA>
9,2024-03,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.16,Brasil,202403,5.8,Rio Grande do Sul,202401,<NA>


In [ ]:
# --- Verificações de qualidade (rode e leia antes de salvar) --------------------------------
print("Dimensões:", base_mensal.shape, "| meses com boletim:", int((base_mensal["n_boletins"] > 0).sum()))
print("\n% de valores vazios por coluna:")
print((base_mensal.isna().mean() * 100).round(1).to_string())

alertas = []
if base_mensal["ipca_variacao_mensal"].isna().all():
    alertas.append("IPCA todo vazio: coleta falhou ou os períodos não intersectam a janela.")
if base_mensal["taxa_desocupacao"].isna().all():
    alertas.append("Desocupação toda vazia: coleta falhou ou os períodos não intersectam a janela.")
if base_mensal["n_boletins"].sum() == 0:
    alertas.append("Nenhum boletim na base mensal: scraping falhou/pulado e a conferência manual está fora da janela?")
if base_mensal["ano_mes"].duplicated().any():
    alertas.append("Há meses duplicados na base mensal (chave ano_mes deveria ser única).")
faltando = [m for m in ["2024-04", "2024-05", "2024-06"] if m not in set(agg_boletins["ano_mes"])]
if faltando:
    alertas.append(f"Sem boletim em meses centrais da enchente de 2024: {faltando}.")

print("\nALERTAS:" if alertas else "\nSem alertas automáticos.")
for a in alertas:
    print(" -", a)


Dimensões: (25, 16) | meses com boletim: 2

% de valores vazios por coluna:
ano_mes                          0.0
n_boletins                       0.0
municipios_afetados             92.0
obitos                          92.0
desaparecidos                   92.0
desalojados                     96.0
pessoas_abrigos                 96.0
pessoas_afetadas                96.0
nivel_guaiba_m                  96.0
ipca_variacao_mensal             0.0
ipca_localidade                  0.0
ipca_periodo_original            0.0
taxa_desocupacao                 0.0
desocupacao_localidade           0.0
desocupacao_periodo_original     0.0
em_crise                        92.0

ALERTAS:
 - Sem boletim em meses centrais da enchente de 2024: ['2024-04', '2024-06'].


> **Nota de limitação (registrar no *dataset card*, item A.5):** a série de scraping está
> concentrada em maio de 2024 (com um único ponto de checagem em abril/2025), enquanto as séries do
> IBGE são contínuas. Para as fases futuras (EDA, séries temporais), o grupo deve ampliar a coleta —
> idealmente com uma fonte estruturada (ex.: nível diário do Guaíba pela ANA/Hidroweb ou pelo
> SGB/CPRM, ou os boletins oficiais da Defesa Civil RS via extração de PDF/OCR) em vez de matérias
> de imprensa esparsas. Isso está documentado como *lacuna conhecida*.

## 7. Base tratada — salvar em CSV e Parquet


In [ ]:
caminho_csv = os.path.join(DIR_TRATADOS, "base_enchentes_indicadores_rs.csv")
caminho_parquet = os.path.join(DIR_TRATADOS, "base_enchentes_indicadores_rs.parquet")
caminho_boletins = os.path.join(DIR_TRATADOS, "base_boletins_enchentes.csv")
caminho_municipios_tratado = os.path.join(DIR_TRATADOS, "municipios_rs.csv")

base_mensal.to_csv(caminho_csv, index=False)
try:
    base_mensal.to_parquet(caminho_parquet, index=False)
except Exception as e:
    print("Não foi possível salvar em Parquet (instale 'pyarrow'):", e)
df_boletins.to_csv(caminho_boletins, index=False)
df_municipios_rs.to_csv(caminho_municipios_tratado, index=False)

print("Salvos em:")
for c in [caminho_csv, caminho_parquet, caminho_boletins, caminho_municipios_tratado]:
    print(" ", c)
print("\nBase mensal (principal):", base_mensal.shape, "| Base por boletim:", df_boletins.shape)


Salvos em:
  ./dados_tratados/base_enchentes_indicadores_rs.csv
  ./dados_tratados/base_enchentes_indicadores_rs.parquet
  ./dados_tratados/base_boletins_enchentes.csv
  ./dados_tratados/municipios_rs.csv

Base mensal (principal): (25, 16) | Base por boletim: (10, 20)


## 8. Registro de proveniência (item 3.4) — salvar log completo e empacotar

In [ ]:
df_provenance = pd.DataFrame(provenance_log)
caminho_provenance = os.path.join(BASE_DIR, "registro_proveniencia.csv")
df_provenance.to_csv(caminho_provenance, index=False)
print(f"{len(df_provenance)} eventos registrados em {caminho_provenance}")
df_provenance


28 eventos registrados em ./registro_proveniencia.csv


,fonte,url,data_hora_coleta_utc,metodo,parametros,observacao,arquivo_bruto,sha256
0,robots.txt de sul21.com.br,https://sul21.com.br/robots.txt,2026-09-19T19:57:30+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
1,robots.txt de diariodonordeste.verdesmares.com.br,https://diariodonordeste.verdesmares.com.br/ro...,2026-09-19T19:57:30+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
2,robots.txt de ihu.unisinos.br,https://ihu.unisinos.br/robots.txt,2026-09-19T19:57:31+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
3,robots.txt de www.correiobraziliense.com.br,https://www.correiobraziliense.com.br/robots.txt,2026-09-19T19:57:32+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
4,robots.txt de www.revistaoeste.com,https://www.revistaoeste.com/robots.txt,2026-09-19T19:57:32+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
5,Sul21,https://sul21.com.br/?p=235018,2026-09-19T19:57:33+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_enchentes/2024-05-01__...,9d8fcbd5727d4329c194ea1f9734240d87e17f2e1f1544...
6,Sul21,https://sul21.com.br/?p=235344,2026-09-19T19:57:37+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_enchentes/2024-05-02__...,2471cca2e2c7a3cc4b0bf33546970669037ec376bdc0b6...
7,Diário do Nordeste,https://diariodonordeste.verdesmares.com.br/ul...,2026-09-19T19:57:40+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_enchentes/2024-05-02_r...,359101a7e0cf2dc5e3ce2fe3dad41e347f703c1ed183de...
8,Sul21,https://sul21.com.br/?p=235669,2026-09-19T19:57:43+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_enchentes/2024-05-03__...,0350930475f8c6bb3e82a913f5268e24ffc66548dcb1d9...
9,Sul21,https://sul21.com.br/?p=237057,2026-09-19T19:57:47+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_enchentes/2024-05-08__...,860feedb82e19762bd982f29347ec9496094dbce3f26a4...


In [ ]:
# Empacota a entrega. O Colab apaga o disco ao desconectar: baixe o zip (ou use MONTAR_DRIVE=True).
# HTML bruto de matérias é conteúdo protegido por direitos autorais: por padrão fica FORA do zip.
caminho_zip = os.path.join(BASE_DIR, "entrega_trabalho1.zip")
with zipfile.ZipFile(caminho_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(caminho_provenance, "registro_proveniencia.csv")
    for pasta in [DIR_TRATADOS, DIR_BRUTOS]:
        for raiz, _, arquivos in os.walk(pasta):
            for a in arquivos:
                if a.endswith(".html") and not INCLUIR_HTML_BRUTO_NO_ZIP:
                    continue
                caminho = os.path.join(raiz, a)
                z.write(caminho, os.path.relpath(caminho, BASE_DIR))
print("Zip criado:", caminho_zip, f"({os.path.getsize(caminho_zip) / 1024:.0f} KB)")

try:
    from google.colab import files
    files.download(caminho_zip)
except ImportError:
    print("(fora do Colab: pegue o zip no caminho acima)")


Zip criado: ./entrega_trabalho1.zip (32 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Postura ética e legal (item 3.5)

- **robots.txt:** verificado programaticamente para **cada URL** (Seção 2). O resultado desta
  execução está em `df_robots`; o resumo é impresso na célula abaixo — **copie-o para o dataset
  card (A.6)**. URLs `bloqueado`/`indeterminado` foram puladas (salvo decisão explícita do grupo).
- **Licença / termos de uso de cada fonte:**
  - *Portais de notícia* (Sul21, Correio Braziliense, IHU/Unisinos, Diário do Nordeste, Revista
    Oeste): conteúdo com direitos autorais jornalísticos. **Não reproduzimos o texto das
    matérias** na base tratada — extraímos apenas **fatos numéricos** (nº de municípios, óbitos,
    desalojados etc.), com o link da fonte para atribuição. O HTML bruto é guardado apenas
    para auditoria/reprodutibilidade e **não deve ser redistribuído** (repositório público, entrega
    com HTMLs): por isso fica fora do zip por padrão. Os trechos curtos em `evidencias`
    (`df_scraping_raw`) servem à conferência interna — se a base for publicada, remova essa coluna.
  - *API do IBGE* (Localidades e SIDRA): dados públicos abertos, disponibilizados pelo próprio órgão
    federal para reuso, inclusive acadêmico ([Portal de Serviços do IBGE](https://servicodados.ibge.gov.br/api/docs)).
- **Dados pessoais / LGPD:** a base não contém dados pessoais identificáveis — os números
  ("pessoas afetadas", "óbitos") são agregados populacionais, sem nomes ou identificação individual.
  Não se aplica minimização/anonimização adicional.
- **Não sobrecarregar servidores:** delay de `DELAY_ENTRE_REQUISICOES` (ou o `Crawl-delay` do site,
  se maior) entre requisições de scraping, sem paralelismo; retries limitados (2–3 tentativas com
  espera crescente). O total de requisições fica no log de proveniência.


In [ ]:
# Resumo para copiar no dataset card (A.6) — gerado a partir do que realmente aconteceu nesta execução
print("robots.txt — resultado por URL:")
print(df_robots["resultado"].value_counts().to_string())
puladas = df_robots[df_robots["resultado"] != "permitido"]
if len(puladas):
    print("\nURLs NÃO raspadas (bloqueadas/indeterminadas):")
    for _, r in puladas.iterrows():
        print(f" - {r['url']}  [{r['resultado']}; {r['detalhe']}]")
print("\nRequisições registradas no log:", len(provenance_log))


robots.txt — resultado por URL:
resultado
permitido    10

Requisições registradas no log: 28


## 10. Dataset Card (Apêndice A)

### A.1 Identificação
- **Nome da base:** Enchentes RS 2024 × Indicadores Econômicos
- **Grupo / integrantes:** _(preencher)_
- **Tema e pergunta motivadora:** Relação entre a severidade da enchente de 2024 no Rio Grande do
  Sul (nº de municípios afetados, óbitos, desaparecidos, desalojados, pessoas em abrigos, nível do
  Guaíba) e indicadores econômicos (IPCA de Porto Alegre, taxa de desocupação), permitindo investigar
  se o desastre climático coincide com deterioração de indicadores econômicos.
- **Data da coleta:** ver `registro_proveniencia.csv` (coluna `data_hora_coleta_utc`)

### A.2 Fontes e proveniência
- **Fonte 1 — nome e URLs:** boletins/matérias sobre as enchentes de 2024 (ver `URLS_ENCHENTES`:
  Sul21, Correio Braziliense, IHU/Unisinos, Diário do Nordeste e Revista Oeste).
- **Fonte 1 — método:** Web scraping (`requests` + `BeautifulSoup`), regex com validação de faixa
  (inclusive concordância de gênero pt-BR para separar "municípios afetados" de "pessoas afetadas")
  + conferência manual (que tem prioridade; origem de cada valor em `origem_*`).
- **Fonte 1 — licença/termos:** conteúdo jornalístico; usamos apenas fatos numéricos, com atribuição de URL.
- **Fonte 2 — nome e URL:** API do IBGE — Localidades (`servicodados.ibge.gov.br/api/v1/localidades`)
  e SIDRA/Agregados (`servicodados.ibge.gov.br/api/v3/agregados`): tabela 7060 (IPCA) e tabela 4099
  (PNAD Contínua trimestral — taxa de desocupação).
- **Fonte 2 — método:** API REST (JSON).
- **Fonte 2 — licença/termos:** dados públicos abertos do IBGE.
- **Chave de integração:** `ano_mes` (data do boletim arredondada para o mês, casada com a
  granularidade mensal do IPCA e com os meses de cada trimestre da PNAD).

### A.3 Dicionário de variáveis (`base_enchentes_indicadores_rs.csv` — base mensal)

| Variável | Tipo | Descrição | Unidade |
|---|---|---|---|
| ano_mes | categórica | Mês de referência (chave; um registro por mês da janela) | AAAA-MM |
| n_boletins | numérica discreta | Nº de boletins/matérias coletados no mês | contagem |
| municipios_afetados | numérica discreta | Nº de municípios do RS afetados pela enchente (máx. do mês) | contagem (0-497) |
| obitos | numérica discreta | Óbitos confirmados (máx. do mês) | contagem |
| desaparecidos | numérica discreta | Pessoas desaparecidas (máx. do mês) | contagem |
| desalojados | numérica contínua | Pessoas desalojadas (máx. do mês) | pessoas |
| pessoas_abrigos | numérica contínua | Pessoas em abrigos públicos (máx. do mês) | pessoas |
| pessoas_afetadas | numérica contínua | Total de pessoas afetadas (máx. do mês) | pessoas |
| nivel_guaiba_m | numérica contínua | Nível do Guaíba em Porto Alegre (máx. do mês) | metros |
| ipca_variacao_mensal | numérica contínua | Variação mensal do IPCA na localidade de `ipca_localidade` | % |
| ipca_localidade | categórica | Localidade efetivamente usada no IPCA (Porto Alegre, na prática) | - |
| ipca_periodo_original | categórica | Código do período no SIDRA | AAAAMM |
| taxa_desocupacao | numérica contínua | Taxa de desocupação (PNAD Contínua; **trimestral repetida nos 3 meses**) | % |
| desocupacao_localidade | categórica | Localidade da desocupação (RS/UF ou fallback) | - |
| desocupacao_periodo_original | categórica | Trimestre original no SIDRA (AAAA0T) | AAAA01–AAAA04 |
| em_crise | booleana anulável | `True` se `municipios_afetados >= 300`; `<NA>` se não há boletim no mês | - |

`base_boletins_enchentes.csv` (granularidade de boletim): `data_ref`, `ano_mes`, `fonte`, `url`, os 7
campos numéricos, `origem_<campo>` (manual/automatico), `data_publicacao_meta`, `data_divergente`.
`municipios_rs.csv`: `municipio_id`, `municipio_nome`, `municipio_nome_normalizado`.

### A.4 Volume e granularidade
- **Nº de linhas / colunas:** ver a célula de resumo logo abaixo (preencher aqui após rodar).
- **O que representa uma linha:** um **mês** da janela de análise, com o resumo dos boletins do mês
  (quando houver) e os indicadores econômicos do mês.
- **Cobertura:** Rio Grande do Sul (estado) para a enchente; IPCA de Porto Alegre e PNAD do RS (ver
  colunas `*_localidade`); janela `2023-06` a `2025-06` (boletins concentrados em maio/2024, com um
  ponto de checagem em abril/2025).

### A.5 Limitações e decisões
- **Dados descartados:** URLs puladas por `robots.txt` (ver Seção 9); valores extraídos fora da
  faixa plausível são descartados; boletins fora da janela não entram na base mensal.
- **Lacunas conhecidas:** série de scraping concentrada em maio/2024, com um único ponto de
  checagem um ano depois; algumas linhas de `df_boletins` não são observações independentes
  (mesmo boletim oficial citado por veículos diferentes no mesmo dia); desocupação trimestral
  repetida por mês; conferência manual a ser revalidada pelo grupo.
- **Decisões de limpeza relevantes:** manual > automático (com `origem_*`); agregação mensal por
  **máximo** em todos os campos (valores maiores = pior situação, inclusive `nivel_guaiba_m`);
  `em_crise` com limiar arbitrário (300 de 497 municípios) e sem transformar ausência em `False`;
  nomes de município normalizados com `unidecode`.

### A.6 Considerações éticas
- **Contém dados pessoais?** Não.
- **Restrições de uso/redistribuição:** ver Seção 9 (não redistribuir o HTML/texto das matérias;
  dados do IBGE são abertos).
- **robots.txt verificado?** Sim, por URL — copiar o resumo da célula da Seção 9.


In [ ]:
# Resumo para preencher o A.4 do dataset card
print("Base mensal:", base_mensal.shape[0], "linhas x", base_mensal.shape[1], "colunas")
print("Base por boletim:", df_boletins.shape[0], "linhas x", df_boletins.shape[1], "colunas")
print("Período:", base_mensal["ano_mes"].min(), "a", base_mensal["ano_mes"].max())
print("Meses com boletim:", ", ".join(base_mensal.loc[base_mensal["n_boletins"] > 0, "ano_mes"]))


Base mensal: 25 linhas x 16 colunas
Base por boletim: 10 linhas x 20 colunas
Período: 2023-06 a 2025-06
Meses com boletim: 2024-05, 2025-04


## 11. Próximos passos (fases futuras da disciplina)

- Trocar/complementar as matérias por **fonte estruturada** do nível do Guaíba (ANA/Hidroweb;
  SGB/CPRM) e pelos boletins oficiais da Defesa Civil RS (`estado.rs.gov.br`), via extração de
  PDF/OCR, para uma série diária em vez de pontos esparsos de imprensa concentrados em maio/2024.
- Estender a coleta com pontos de checagem intermediários entre maio/2024 e abril/2025 (ex.:
  balanços de 6 meses, 1 ano da reconstrução).
- Buscar população municipal (estimativas do IBGE) e o PIB municipal para enriquecer
  `municipios_rs.csv` e permitir análises per capita / por porte de município.
- Adicionar dados de precipitação (INMET) e de área atingida por inundação (MapBiomas/Ipea, como no
  relatório do IPEA citado nas fontes de apoio) como novas fontes.
- Nas fases de EDA/regressão/classificação: usar `em_crise` como rótulo **sem** `municipios_afetados`
  como preditor; e considerar a defasagem temporal entre a enchente e os indicadores econômicos.
